# 10 · 03 — Model: can the spec's channels predict the outcome?

## The protocol, and why it is not the obvious one

Session 08 learned this the expensive way: **scoring a job at a fixed lead before its end
leaks its runtime.** At `lead` minutes before the end the model is handed
`elapsed = run_time − lead`, and with `time_limit` also present it recovers
`run_time / time_limit` — and 98.6 % of `TIMEOUT` jobs sit at their limit. Average
precision went from 0.699 to 0.92 without predicting anything.

So every job here is scored **once, at a fixed elapsed time `T` since it started**, using
only the 20 minutes of telemetry ending at `T`. Nothing about the end is used. The cohort
at `T` is "jobs still running at `T`", which is what a scheduler sees.

## Arms

A telemetry number means nothing without a context baseline, so every telemetry arm is
reported against one:

| arm | features |
|---|---|
| **S** | size only — nodes, cores, GPUs |
| **M** | metadata: size + `time_limit`, `mem_req`, tasks, priority, qos, partition |
| **P** | telemetry only — the spec's channels, 7 summaries each |
| **M+P** | both |
| **M+P+U** | both, plus user identity |

`U` is separated deliberately. Session 08 found the whole telemetry separation was user
identity; if `M+P+U` beats `M+P` by a lot, the model is learning who submits, not what
is happening.

In [1]:
import json, os, sys
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from lightgbm import LGBMClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

sys.path.insert(0, os.path.abspath("."))
import exalib as X

J = pd.read_parquet("data/jobs.parquet")
z = np.load("data/series.npz")
CH = [k[3:] for k in z.files if k.startswith("v__")]
V = {c: z[f"v__{c}"] for c in CH}
O = {c: z[f"o__{c}"] for c in CH}
K = np.diff(O[CH[0]])

WIN  = 20                                # minutes of telemetry
TS   = [20, 40, 60]                      # elapsed minutes at which the job is scored
# gv100card0/3 are identically zero in 2020-05, so no GPU-power channel exists here.
PCH  = ["gpu_temp_max", "gpu0_mtemp", "node_pow", "fan", "ambient",
        "mem_active", "swap_free", "proc_run", "cpu_wio", "load_one"]
LGB  = dict(objective="binary", learning_rate=0.05, num_leaves=31, n_estimators=300,
            min_child_samples=30, subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
            random_state=X.SEED, deterministic=True, force_row_wise=True, verbose=-1, n_jobs=4)
print(f"{len(J):,} jobs, {len(PCH)} channels x {len(X.AGGS)} aggregates")

69,096 jobs, 10 channels x 7 aggregates


## Step 1 — metadata design matrix

`run_time`, `end_time` and anything derived from them are absent by construction. The
assertion below is the guard.

In [2]:
J["qos_code"] = J.qos.astype("category").cat.codes
J["part_code"] = J.partition.astype("category").cat.codes
J["user_code"] = J.user_id.astype("category").cat.codes
S_COLS = ["num_nodes_alloc", "num_cores_alloc", "num_gpus_alloc"]
M_COLS = S_COLS + ["time_limit", "mem_req", "num_tasks", "cores_per_task", "priority",
                   "qos_code", "part_code"]
BANNED = {"run_time", "end_time", "start_time", "job_state", "y", "elapsed"}
assert not (set(M_COLS) & BANNED), "leak: an end-relative column entered the metadata arm"
print("metadata columns:", len(M_COLS))

metadata columns: 10


## Step 2 — telemetry features at a fixed elapsed time

In [3]:
def feats_at(T):
    """Rows for the jobs still running at T minutes; window is [T-WIN, T]."""
    live = K >= T
    idx = np.where(live)[0]
    F = np.full((len(idx), len(PCH) * len(X.AGGS)), np.nan, np.float32)
    for r, i in enumerate(idx):
        for j, c in enumerate(PCH):
            o = O[c][i]
            F[r, j * len(X.AGGS):(j + 1) * len(X.AGGS)] = X.window_feats(
                V[c][o + T - WIN:o + T])
    cols = [f"{c}__{a}" for c in PCH for a in X.AGGS]
    return idx, pd.DataFrame(F, columns=cols)

def evaluate(T, target="any"):
    idx, P = feats_at(T)
    d = J.iloc[idx].reset_index(drop=True)
    y = (d.job_state.isin(X.FAILURE_STATES) if target == "any"
         else d.job_state.isin(X.HARDWARE_STATES)).astype(int).values
    ALL = pd.concat([d[M_COLS + ["user_code"]].reset_index(drop=True), P], axis=1)
    cut = d.end_time.quantile(0.8)
    tr, te = (d.end_time <= cut).values, (d.end_time > cut).values
    arms = {"S  size": S_COLS, "M  metadata": M_COLS, "P  telemetry": list(P.columns),
            "M+P": M_COLS + list(P.columns), "M+P+U": M_COLS + list(P.columns) + ["user_code"]}
    base = y[te].mean()
    rows, models = [], {}
    for name, cols in arms.items():
        m = LGBMClassifier(**LGB).fit(ALL.loc[tr, cols], y[tr])
        p = m.predict_proba(ALL.loc[te, cols])[:, 1]
        ap = average_precision_score(y[te], p)
        k = 200 if te.sum() > 400 else max(20, te.sum() // 10)
        top = np.argsort(-p)[:k]
        rows.append(dict(T=T, target=target, arm=name, n_test=int(te.sum()), base=base,
                         AP=ap, lift=ap / base, ROC=roc_auc_score(y[te], p),
                         prec_at_k=y[te][top].mean(), k=k))
        models[name] = (m, cols, p, y[te], d[te])
    return pd.DataFrame(rows), models

## Step 3 — the sweep over T

In [4]:
res, keep = [], {}
for T in TS:
    for tgt in ["any", "hardware"]:
        r, mo = evaluate(T, tgt)
        res.append(r)
        if T == TS[0]:
            keep[tgt] = mo
R = pd.concat(res, ignore_index=True)
for tgt in ["any", "hardware"]:
    print(f"\n=== target = {tgt} ===")
    t = R[R.target == tgt].pivot(index="arm", columns="T",
                                 values=["AP", "lift", "prec_at_k"])
    print(t.round(3).to_string())
print("\nbase rate and test size by T:")
print(R[R.arm == "S  size"][["T", "target", "n_test", "base", "k"]].round(4).to_string(index=False))


=== target = any ===
                 AP                 lift               prec_at_k           
T                20     40     60     20     40     60        20   40    60
arm                                                                        
M  metadata   0.546  0.591  0.636  1.998  1.917  1.964     1.000  1.0  1.00
M+P           0.587  0.619  0.654  2.147  2.009  2.017     1.000  1.0  1.00
M+P+U         0.589  0.642  0.687  2.156  2.083  2.120     0.995  1.0  1.00
P  telemetry  0.506  0.568  0.582  1.852  1.841  1.796     0.995  1.0  0.99
S  size       0.435  0.474  0.500  1.592  1.538  1.544     1.000  1.0  1.00

=== target = hardware ===
                 AP                 lift               prec_at_k              
T                20     40     60     20     40     60        20     40     60
arm                                                                           
M  metadata   0.324  0.386  0.424  1.787  1.980  2.229     0.430  0.505  0.410
M+P           0.352  0.393 

## Step 4 — is the telemetry adding anything over context?

The only comparison that matters: **M+P against M**. Telemetry earns its place only if it
moves that number.

In [5]:
piv = R[R.target == "any"].pivot(index="T", columns="arm", values="AP")
piv["P over M"] = piv["P  telemetry"] / piv["M  metadata"] - 1
piv["M+P over M"] = piv["M+P"] / piv["M  metadata"] - 1
piv["+U over M+P"] = piv["M+P+U"] / piv["M+P"] - 1
print(piv[["M  metadata", "P  telemetry", "M+P", "M+P+U",
           "P over M", "M+P over M", "+U over M+P"]].round(3).to_string())

arm  M  metadata  P  telemetry    M+P  M+P+U  P over M  M+P over M  +U over M+P
T                                                                              
20         0.546         0.506  0.587  0.589    -0.073       0.074        0.004
40         0.591         0.568  0.619  0.642    -0.040       0.048        0.037
60         0.636         0.582  0.654  0.687    -0.085       0.027        0.051


## Step 5 — what the model flags

Session 08's `M3` failed here: metadata's top-100 was only 58 % `TIMEOUT`, because it
cannot tell "about to hit the limit" from "will finish just in time". The composition of
the alert list says what the detector actually is.

In [6]:
for arm in ["M  metadata", "P  telemetry", "M+P"]:
    m, cols, p, yt, dte = keep["any"][arm]
    k = min(200, len(p) // 5)
    top = dte.iloc[np.argsort(-p)[:k]]
    comp = top.job_state.value_counts(normalize=True).round(3).to_dict()
    print(f"{arm:14s} top-{k}: " + "  ".join(f"{a} {b:.0%}" for a, b in comp.items()))

M  metadata    top-200: TIMEOUT 100%
P  telemetry   top-200: TIMEOUT 99%  FAILED 0%  COMPLETED 0%
M+P            top-200: TIMEOUT 100%


In [7]:
m, cols, p, yt, dte = keep["any"]["M+P"]
imp = pd.Series(m.booster_.feature_importance("gain"), index=cols).sort_values(ascending=False)
grp = imp.groupby(lambda c: c.split("__")[0] if "__" in c else "metadata:" + c).sum()
grp = (grp / grp.sum()).sort_values(ascending=False)
print("\nM+P gain share by source\n")
print(grp.head(14).round(4).to_string())


M+P gain share by source

metadata:time_limit         0.1979
node_pow                    0.1437
metadata:priority           0.1133
metadata:num_tasks          0.1084
metadata:mem_req            0.0842
metadata:num_cores_alloc    0.0512
gpu0_mtemp                  0.0483
mem_active                  0.0476
gpu_temp_max                0.0412
load_one                    0.0374
metadata:cores_per_task     0.0301
proc_run                    0.0227
ambient                     0.0208
metadata:num_gpus_alloc     0.0198


## Step 6 — figure

In [8]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
for arm in ["S  size", "M  metadata", "P  telemetry", "M+P", "M+P+U"]:
    s = R[(R.target == "any") & (R.arm == arm)]
    ax[0].plot(s["T"], s.lift, marker="o", label=arm)
ax[0].axhline(1, color="k", ls="--", lw=.8)
ax[0].set_xlabel("elapsed minutes at which the job is scored")
ax[0].set_ylabel("lift = AP / base rate"); ax[0].set_title("early triage · any failure")
ax[0].legend(fontsize=7)
g = grp.head(12)[::-1]
ax[1].barh(np.arange(len(g)), g.values)
ax[1].set_yticks(np.arange(len(g))); ax[1].set_yticklabels(g.index, fontsize=7)
ax[1].set_xlabel("share of total gain"); ax[1].set_title("M+P · where the model looks")
plt.tight_layout(); plt.savefig("figures/03_model.png", dpi=130)
R.to_csv("data/model_results.csv", index=False)
json.dump({"results": R.round(4).to_dict("records"),
           "gain_by_source": grp.round(4).to_dict()},
          open("data/model.json", "w"), indent=1)
print("wrote figures/03_model.png, data/model_results.csv")

wrote figures/03_model.png, data/model_results.csv
